# Aerial Imagery Segmentation Pipeline
## AIRS-Inspired FPN/PSPNet with ResNet Backbone — Binary Segmentation

---

## ⚠️ CRITICAL: Label Mismatch — Read Before Running

### Your current dataset has **solar-panel masks**, but your goal is **rooftop segmentation**.

These are **fundamentally different** segmentation targets:

| Property | Solar-Panel Mask | Rooftop Mask |
|---|---|---|  
| Coverage | Only pixels with solar panels | All roof pixels regardless of panels |
| Sparsity | Very sparse — most roofs have no panels | Denser — most buildings have roofs |
| Shape | Rectangular panel clusters | Complex polygonal roof shapes |
| Overlap | Always a subset of rooftops | Rooftops are the superset |

### ❌ What happens if you train rooftop segmentation on solar-panel masks:
- The model will learn to **ignore most roof surfaces** (they appear as background in solar-panel masks)
- The model will learn to detect **only the small subset of roofs that carry solar panels**
- For roofs without solar panels, the model will predict **no segmentation at all** — not rooftop, not solar panel, just dark/background
- Your IoU for rooftop segmentation will be extremely low because the model predicts a tiny fraction of the actual roof area
- This is a **systematic bias**, not a noise issue — the model will confidently be wrong

---

## 🗺️ Your Options (Evaluated)

**Option A — Train solar-panel segmentation now** ✅ *Recommended starting point*  
Use your current masks to train a high-quality solar-panel detector. This:
- Gives you a working, validated pipeline immediately
- Produces a trained encoder backbone that understands aerial imagery
- Can later be fine-tuned for rooftop segmentation with minimal new code (just swap the dataset)

**Option B — Generate rooftop pseudo-labels then retrain**  
Use a pretrained off-the-shelf rooftop model (e.g., from SpaceNet challenge) to generate pseudo-labels on your 5,000 images, then train on those labels. Quality depends on the quality of the pseudo-label model — can introduce systematic noise.

**Option C — Transfer learning from a rooftop dataset**  
Fine-tune from a pretrained checkpoint trained on SpaceNet, Inria, or WHU Building Dataset — all have rooftop/building footprint masks. This is the most principled path if you can procure the weights.

**Option D — Weakly supervised / semi-supervised**  
Use bounding-box level annotations or image-level labels if pixel-level labels are unavailable. More complex to implement.

---

## ✅ This Notebook Implements: Option A + B-Ready Architecture

**We train a solar-panel segmentation model now.** The code is structured so that switching to rooftop segmentation requires changing only the dataset path and mask folder — everything else stays identical. At the bottom, we describe exactly how to upgrade to rooftop masks.

---

## Step 0: Install Dependencies

In [ ]:
!pip install segmentation-models-pytorch albumentations -q

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Imports and Configuration

In [ ]:
import os
import random
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import train_test_split

# ─────────────────────────────────────────────
#  GLOBAL CONFIG  — edit these to switch targets
# ─────────────────────────────────────────────
CFG = {
    # Paths
    'BASE_DIR'      : '/content/drive/MyDrive/project',
    'IMAGE_DIR'     : '/content/drive/MyDrive/project/images',
    'MASK_DIR'      : '/content/drive/MyDrive/project/masks',
    'METADATA_CSV'  : '/content/drive/MyDrive/project/metadata.csv',
    'CKPT_DIR'      : '/content/drive/MyDrive/project/checkpoints',

    # Data
    'IMG_SIZE'      : 512,       # resize both axes; for 0.075m GSD AIRS imagery 512 covers ~38m
    'VAL_SPLIT'     : 0.15,
    'SEED'          : 42,

    # Model
    'ARCH'          : 'FPN',     # 'FPN' | 'PSPNet' — change here to switch architecture
    'ENCODER'       : 'resnet50',
    'ENCODER_WEIGHTS': 'imagenet',
    'IN_CHANNELS'   : 3,
    'NUM_CLASSES'   : 1,         # binary segmentation

    # Training
    'BATCH_SIZE'    : 8,
    'NUM_WORKERS'   : 2,
    'EPOCHS'        : 50,
    'LR'            : 3e-4,
    'PATIENCE'      : 7,         # early stopping patience

    # Inference
    'THRESHOLD'     : 0.5,       # sigmoid output threshold for binary mask

    # Mask convention
    # Assumption: masks are single-channel PNG where:
    #   0   = background
    #   255 = foreground (solar panel / roof)
    # The Dataset normalizes to 0/1 automatically.
    'MASK_SCALE'    : 255,

    # Target label — used only for print/display, no code changes needed
    'TARGET_LABEL'  : 'Solar Panel',   # ← change to 'Rooftop' when you get rooftop masks
}

os.makedirs(CFG['CKPT_DIR'], exist_ok=True)

# Reproducibility
random.seed(CFG['SEED'])
np.random.seed(CFG['SEED'])
torch.manual_seed(CFG['SEED'])
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'Target label: {CFG["TARGET_LABEL"]}')

## Step 3: Load and Split Metadata

In [ ]:
df = pd.read_csv(CFG['METADATA_CSV'])

# Validate expected columns
assert 'image_name' in df.columns, "CSV must have 'image_name' column"
assert 'mask_name'  in df.columns, "CSV must have 'mask_name' column"

print(f'Total samples: {len(df)}')
print(df.head())

# Train / validation split (stratification not needed for segmentation)
train_df, val_df = train_test_split(
    df,
    test_size=CFG['VAL_SPLIT'],
    random_state=CFG['SEED'],
    shuffle=True
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f'\nTrain: {len(train_df)}  |  Val: {len(val_df)}')

## Step 4: Albumentations Transforms

**Class imbalance note**: Solar-panel masks are highly sparse (positive class << background). We address this via:
1. `RandomCrop` during training to increase the chance of cropping a region containing the target
2. Combined Binary Cross-Entropy + Dice loss (Dice is naturally robust to imbalance)
3. Optional: set `pos_weight` in BCEWithLogitsLoss if imbalance is severe

In [ ]:
SZ = CFG['IMG_SIZE']

train_transforms = A.Compose([
    A.Resize(SZ, SZ),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.4),
    A.OneOf([
        A.GaussNoise(var_limit=(5, 30), p=1),
        A.GaussianBlur(blur_limit=(3, 5), p=1),
    ], p=0.3),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05, p=0.4),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_transforms = A.Compose([
    A.Resize(SZ, SZ),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

## Step 5: Custom Dataset

In [ ]:
class SegmentationDataset(Dataset):
    """
    Generic binary segmentation dataset.

    Assumptions:
    - Images: 3-channel RGB, any format PIL can open
    - Masks : single-channel PNG, values 0 (bg) or 255 (fg)
              (change CFG['MASK_SCALE'] = 1 if masks are already 0/1)
    """

    def __init__(self, df, image_dir, mask_dir, transform=None, mask_scale=255):
        self.df          = df
        self.image_dir   = image_dir
        self.mask_dir    = mask_dir
        self.transform   = transform
        self.mask_scale  = mask_scale

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # --- Load image ---
        img_path = os.path.join(self.image_dir, row['image_name'])
        image = np.array(Image.open(img_path).convert('RGB'), dtype=np.uint8)

        # --- Load mask ---
        msk_path = os.path.join(self.mask_dir, row['mask_name'])
        mask = np.array(Image.open(msk_path).convert('L'), dtype=np.float32)

        # Normalise mask to [0, 1]
        if self.mask_scale > 1:
            mask = (mask / self.mask_scale).astype(np.float32)
        mask = np.clip(mask, 0, 1)          # safety clamp

        # --- Apply transforms ---
        if self.transform:
            aug = self.transform(image=image, mask=mask)
            image = aug['image']             # (C, H, W) tensor — float32, normalised
            mask  = aug['mask']              # (H, W)   tensor

        mask = mask.unsqueeze(0)             # → (1, H, W)
        return image, mask


# Instantiate datasets
train_ds = SegmentationDataset(train_df, CFG['IMAGE_DIR'], CFG['MASK_DIR'],
                               transform=train_transforms, mask_scale=CFG['MASK_SCALE'])
val_ds   = SegmentationDataset(val_df,   CFG['IMAGE_DIR'], CFG['MASK_DIR'],
                               transform=val_transforms,   mask_scale=CFG['MASK_SCALE'])

# DataLoaders
train_loader = DataLoader(train_ds, batch_size=CFG['BATCH_SIZE'],
                          shuffle=True,  num_workers=CFG['NUM_WORKERS'], pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG['BATCH_SIZE'],
                          shuffle=False, num_workers=CFG['NUM_WORKERS'], pin_memory=True)

print(f'Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}')

## Step 6: Sanity Check — Visualise a Sample

In [ ]:
def denorm(tensor, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)):
    """Reverse ImageNet normalisation for display."""
    t = tensor.clone().permute(1, 2, 0).cpu().numpy()
    t = t * np.array(std) + np.array(mean)
    return np.clip(t, 0, 1)

imgs, masks = next(iter(train_loader))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(4):
    axes[0, i].imshow(denorm(imgs[i]))
    axes[0, i].set_title(f'Image {i}'); axes[0, i].axis('off')
    axes[1, i].imshow(masks[i, 0].cpu().numpy(), cmap='gray', vmin=0, vmax=1)
    axes[1, i].set_title(f'{CFG["TARGET_LABEL"]} Mask {i}'); axes[1, i].axis('off')
plt.tight_layout()
plt.show()

## Step 7: Model Definition (FPN or PSPNet with ResNet Backbone)

Set `CFG['ARCH'] = 'PSPNet'` to switch. Architecture is inspired by the AIRS paper which uses FPN/PSPNet + ResNet for high-resolution aerial images.

In [ ]:
def build_model(cfg):
    arch    = cfg['ARCH'].upper()
    kwargs  = dict(
        encoder_name    = cfg['ENCODER'],
        encoder_weights = cfg['ENCODER_WEIGHTS'],
        in_channels     = cfg['IN_CHANNELS'],
        classes         = cfg['NUM_CLASSES'],
        activation      = None,   # raw logits — we apply sigmoid manually or in loss
    )

    if arch == 'FPN':
        model = smp.FPN(**kwargs)
    elif arch == 'PSPNET':
        # PSPNet requires image size divisible by 8
        model = smp.PSPNet(**kwargs)
    else:
        raise ValueError(f'Unknown arch: {arch}. Choose FPN or PSPNet.')

    return model


model = build_model(CFG).to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Architecture   : {CFG["ARCH"]} + {CFG["ENCODER"]}')
print(f'Total params   : {total_params:,}')
print(f'Trainable params: {trainable_params:,}')

## Step 8: Loss Function, Metrics, Optimiser

We use **BCE + Dice** combined loss which is robust to class imbalance.

In [ ]:
# ─── Loss ───────────────────────────────────────────────────────────
# Option: set pos_weight to handle heavy imbalance.
# e.g. if solar panels cover ~5% of pixels, pos_weight ≈ 19
# Uncomment and adjust if needed:
# bce_loss = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([19.0]).to(DEVICE))

bce_loss  = nn.BCEWithLogitsLoss()
dice_loss = smp.losses.DiceLoss(mode='binary', from_logits=True)

def combined_loss(logits, targets, bce_w=0.5, dice_w=0.5):
    return bce_w * bce_loss(logits, targets) + dice_w * dice_loss(logits, targets)


# ─── Metrics ────────────────────────────────────────────────────────
def compute_metrics(logits, targets, threshold=0.5):
    """
    Returns IoU and Dice for a batch.
    logits : (B, 1, H, W) float  — raw model output
    targets: (B, 1, H, W) float  — binary 0/1
    """
    preds = (torch.sigmoid(logits) > threshold).float()

    # Intersection / Union / Dice
    inter = (preds * targets).sum(dim=(1, 2, 3))     # per sample
    union = preds.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3)) - inter

    iou  = ((inter + 1e-6) / (union + 1e-6)).mean().item()
    dice = ((2 * inter + 1e-6) / (preds.sum(dim=(1, 2, 3)) +
                                   targets.sum(dim=(1, 2, 3)) + 1e-6)).mean().item()
    return iou, dice


# ─── Optimiser + Scheduler ─────────────────────────────────────────
optimiser = torch.optim.AdamW(model.parameters(), lr=CFG['LR'], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimiser, T_max=CFG['EPOCHS'], eta_min=1e-6
)

print('Loss, metrics, and optimiser ready.')

## Step 9: Training + Validation Loop

In [ ]:
# ─── History ──────────────────────────────────────────────────────
history = {'train_loss': [], 'val_loss': [], 'val_iou': [], 'val_dice': []}

# ─── Early stopping state ─────────────────────────────────────────
best_val_iou    = -1.0
patience_count  = 0
best_ckpt_path  = os.path.join(CFG['CKPT_DIR'], 'best_model.pth')


def train_one_epoch(model, loader, optimiser):
    model.train()
    total_loss = 0.0
    for imgs, masks in loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimiser.zero_grad()
        logits = model(imgs)
        loss   = combined_loss(logits, masks)
        loss.backward()
        optimiser.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def validate(model, loader):
    model.eval()
    total_loss = total_iou = total_dice = 0.0
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            logits       = model(imgs)
            loss         = combined_loss(logits, masks)
            iou, dice    = compute_metrics(logits, masks, CFG['THRESHOLD'])
            total_loss  += loss.item()
            total_iou   += iou
            total_dice  += dice
    n = len(loader)
    return total_loss / n, total_iou / n, total_dice / n


# ─── Main loop ───────────────────────────────────────────────────
print(f'Starting training for {CFG["EPOCHS"]} epochs...\n')

for epoch in range(1, CFG['EPOCHS'] + 1):
    train_loss               = train_one_epoch(model, train_loader, optimiser)
    val_loss, val_iou, val_dice = validate(model, val_loader)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_iou'].append(val_iou)
    history['val_dice'].append(val_dice)

    print(f'Epoch [{epoch:03d}/{CFG["EPOCHS"]}]  '
          f'Train Loss: {train_loss:.4f}  '
          f'Val Loss: {val_loss:.4f}  '
          f'Val IoU: {val_iou:.4f}  '
          f'Val Dice: {val_dice:.4f}')

    # --- Checkpoint: save best model ---
    if val_iou > best_val_iou:
        best_val_iou   = val_iou
        patience_count = 0
        torch.save({
            'epoch'     : epoch,
            'model_state': model.state_dict(),
            'optim_state': optimiser.state_dict(),
            'val_iou'   : val_iou,
            'val_dice'  : val_dice,
            'cfg'       : CFG,
        }, best_ckpt_path)
        print(f'  ✅ Best model saved (IoU = {best_val_iou:.4f})')
    else:
        patience_count += 1
        if patience_count >= CFG['PATIENCE']:
            print(f'\n⏹ Early stopping triggered after {epoch} epochs.')
            break

print(f'\nTraining complete. Best Val IoU: {best_val_iou:.4f}')

## Step 10: Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'],   label='Val Loss')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].set_xlabel('Epoch')

# IoU
axes[1].plot(history['val_iou'], color='green', label='Val IoU')
axes[1].set_title('Validation IoU'); axes[1].legend(); axes[1].set_xlabel('Epoch')

# Dice
axes[2].plot(history['val_dice'], color='orange', label='Val Dice')
axes[2].set_title('Validation Dice'); axes[2].legend(); axes[2].set_xlabel('Epoch')

plt.tight_layout()
plt.savefig(os.path.join(CFG['CKPT_DIR'], 'training_curves.png'), dpi=150)
plt.show()

## Step 11: Load Best Checkpoint + Single-Image Inference

In [ ]:
# ─── Load best checkpoint ─────────────────────────────────────────
checkpoint = torch.load(best_ckpt_path, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state'])
model.eval()
print(f'Loaded best model from epoch {checkpoint["epoch"]} '
      f'with Val IoU = {checkpoint["val_iou"]:.4f}')


# ─── Inference helper ─────────────────────────────────────────────
def predict_single(model, image_path, transform, threshold=0.5):
    """
    Run inference on a single image file.

    Returns
    -------
    image_np   : (H, W, 3) uint8          — original resized image (display)
    prob_map   : (H, W)    float [0,1]    — raw sigmoid probability map
    pred_mask  : (H, W)    uint8 {0,1}   — binary predicted mask
    """
    raw = np.array(Image.open(image_path).convert('RGB'), dtype=np.uint8)

    # Keep the un-normalised resized image for display
    resized = np.array(Image.fromarray(raw).resize(
        (CFG['IMG_SIZE'], CFG['IMG_SIZE']), Image.BILINEAR
    ))

    # Apply val transforms
    aug   = transform(image=raw)
    inp   = aug['image'].unsqueeze(0).to(DEVICE)   # (1, C, H, W)

    with torch.no_grad():
        logit    = model(inp)                        # (1, 1, H, W)
        prob_map = torch.sigmoid(logit).squeeze().cpu().numpy()  # (H, W)

    pred_mask = (prob_map > threshold).astype(np.uint8)
    return resized, prob_map, pred_mask


print('Inference function ready.')

## Step 12: Visualisation — Image | Ground Truth | Prediction | Overlay

In [ ]:
def visualise_prediction(image_path, mask_path, model, transform,
                         threshold=0.5, save_path=None):
    """
    Display a 4-panel figure:
      1. Original aerial image
      2. Ground-truth mask
      3. Predicted mask
      4. Overlay (prediction in red on original image)
    """
    # Run inference
    img_np, prob_map, pred_mask = predict_single(
        model, image_path, transform, threshold
    )

    # Load & resize ground truth for display
    gt_raw  = np.array(Image.open(mask_path).convert('L'))
    gt_mask = np.array(Image.fromarray(gt_raw).resize(
        (CFG['IMG_SIZE'], CFG['IMG_SIZE']), Image.NEAREST
    ))
    gt_mask = (gt_mask / CFG['MASK_SCALE']).clip(0, 1)

    # Overlay: blend red channel into image
    overlay      = img_np.copy().astype(np.float32)
    overlay[..., 0] = np.clip(overlay[..., 0] + pred_mask * 120, 0, 255)
    overlay[..., 1] = np.clip(overlay[..., 1] - pred_mask * 60,  0, 255)
    overlay[..., 2] = np.clip(overlay[..., 2] - pred_mask * 60,  0, 255)
    overlay         = overlay.astype(np.uint8)

    # ─── Plot ─────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    axes[0].imshow(img_np)
    axes[0].set_title('Original Image')
    axes[0].axis('off')

    axes[1].imshow(gt_mask, cmap='gray', vmin=0, vmax=1)
    axes[1].set_title(f'Ground Truth\n({CFG["TARGET_LABEL"]})')
    axes[1].axis('off')

    axes[2].imshow(prob_map, cmap='viridis', vmin=0, vmax=1)
    cb = plt.colorbar(axes[2].images[0], ax=axes[2], fraction=0.046, pad=0.04)
    cb.set_label('Probability')
    axes[2].set_title(f'Predicted Probability\n(threshold={threshold})')
    axes[2].axis('off')

    axes[3].imshow(overlay)
    patch = mpatches.Patch(color='red', label=f'Predicted {CFG["TARGET_LABEL"]}')
    axes[3].legend(handles=[patch], loc='lower right', fontsize=8)
    axes[3].set_title('Overlay')
    axes[3].axis('off')

    plt.suptitle(
        f'Segmentation Result — {os.path.basename(image_path)}',
        fontsize=13, y=1.02
    )
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Saved: {save_path}')
    plt.show()


# ─── Run on a sample from validation set ──────────────────────────
sample   = val_df.iloc[0]
img_file = os.path.join(CFG['IMAGE_DIR'], sample['image_name'])
msk_file = os.path.join(CFG['MASK_DIR'],  sample['mask_name'])

visualise_prediction(
    img_file, msk_file, model, val_transforms,
    threshold = CFG['THRESHOLD'],
    save_path = os.path.join(CFG['CKPT_DIR'], 'sample_prediction.png')
)

## Step 13: Batch Inference on Validation Set (Optional)

In [ ]:
# Visualise first N samples from val set
N = 4

fig, axes = plt.subplots(N, 4, figsize=(20, 5 * N))
col_titles = ['Image', 'Ground Truth', 'Prediction', 'Overlay']

for col, title in enumerate(col_titles):
    axes[0, col].set_title(title, fontsize=12, fontweight='bold')

for i in range(N):
    row    = val_df.iloc[i]
    i_path = os.path.join(CFG['IMAGE_DIR'], row['image_name'])
    m_path = os.path.join(CFG['MASK_DIR'],  row['mask_name'])

    img_np, prob_map, pred = predict_single(model, i_path, val_transforms, CFG['THRESHOLD'])

    gt   = np.array(Image.open(m_path).convert('L').resize(
               (CFG['IMG_SIZE'], CFG['IMG_SIZE']), Image.NEAREST)) / CFG['MASK_SCALE']
    ov   = img_np.copy().astype(np.float32)
    ov[..., 0] = np.clip(ov[..., 0] + pred * 120, 0, 255)
    ov[..., 1] = np.clip(ov[..., 1] - pred * 60,  0, 255)
    ov[..., 2] = np.clip(ov[..., 2] - pred * 60,  0, 255)

    axes[i, 0].imshow(img_np)
    axes[i, 1].imshow(gt,       cmap='gray', vmin=0, vmax=1)
    axes[i, 2].imshow(prob_map, cmap='viridis', vmin=0, vmax=1)
    axes[i, 3].imshow(ov.astype(np.uint8))
    for j in range(4): axes[i, j].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(CFG['CKPT_DIR'], 'batch_predictions.png'), dpi=150)
plt.show()

## 🔁 How to Switch to Rooftop Segmentation

When you have rooftop masks, **only 3 things need to change**:

```python
# 1. Update the mask directory path
CFG['MASK_DIR']     = '/content/drive/MyDrive/project/rooftop_masks'

# 2. Update the display label
CFG['TARGET_LABEL'] = 'Rooftop'

# 3. (Optional) Fine-tune from your solar-panel checkpoint instead of ImageNet
#    Replace encoder_weights='imagenet' in build_model with:
checkpoint = torch.load(best_ckpt_path)
model.load_state_dict(checkpoint['model_state'])
# Then continue training — the encoder already understands aerial imagery structure.
```

Everything else — Dataset, DataLoader, loss, metrics, visualisation — remains identical.

---

## 📊 Where to Get Rooftop Masks

| Source | URL | Notes |
|---|---|---|
| **SpaceNet Challenges** | spacenet.ai | Building footprints, globally diverse |
| **Inria Aerial Image Labeling** | project.inria.fr/aerialimagelabeling | Dense urban rooftops |
| **WHU Building Dataset** | gpcv.whu.edu.cn | 187+ cities, binary building masks |
| **CrowdAI Mapping Challenge** | Kaggle | 300K+ satellite building images |
| **AIRS Dataset** (original paper) | Request from authors | High-resolution aerial, roof-focused |
| **Label your own** | CVAT / Roboflow Annotate / Labelbox | Most accurate for your specific region/imagery |

---

## 📦 What the Final Output Will Look Like

### Right now (with solar-panel masks)

After running this notebook end-to-end, the model will produce:
- **A binary mask per image** where white pixels = *predicted solar panel locations*
- **A probability heatmap** (viridis colormap, 0→blue, 1→yellow) showing confidence per pixel
- **An overlay image** with red highlighting over predicted solar-panel regions
- **Val IoU and Dice scores** on the hold-out validation set

**This is solar-panel segmentation output — NOT rooftop segmentation.** The model will:
- Correctly detect roofs that carry solar panels
- Leave bare rooftops (no panels) as background (black)
- Be **wrong** for any use case that needs all rooftop pixels

### Visual interpretation
- **High-contrast white blob** on prediction = model is confident a solar panel is there
- **Gray patch** on probability map = uncertain region
- **Dark / black** = confident background (no panel)
- **Good result**: prediction blob closely matches ground-truth blob (IoU > 0.6 is reasonable for sparse masks)

### To get true rooftop masks

1. **Procure rooftop labels** from SpaceNet, Inria, WHU, or annotate your own with CVAT
2. **Change 3 config lines** as shown in the section above
3. **Fine-tune from this solar-panel checkpoint** (encoder already understands aerial imagery)
4. After fine-tuning, predictions will show **entire roof polygons** including bare rooftops

### Expected metric targets (for reference)

| Stage | Target | Expected Val IoU |
|---|---|---|
| Solar-panel segmentation (this notebook) | Solar panels | 0.55 – 0.75 |
| Rooftop segmentation (after label upgrade) | Rooftops | 0.70 – 0.85 (AIRS-level) |

---

## 📁 What Gets Saved

After training you will find in `CFG['CKPT_DIR']`:
```
checkpoints/
  best_model.pth          ← Best checkpoint (by Val IoU)
  training_curves.png     ← Loss, IoU, Dice over epochs
  sample_prediction.png   ← 4-panel visual for one sample
  batch_predictions.png   ← 4-panel visual for N val samples
```